# LARC 그래프 수치 정리

원본 실험의 최종 평가 수치와 고정 오버헤드 수치에서 그래프용 표, CSV, PNG, JSON을 만듭니다.

- 기본 실행: 현재 초안의 표 수치로 그래프를 만듭니다.
- 새 실험 반영: `RUN_SOURCE_EXPERIMENT = True`로 바꾸면 원본 노트북을 실행해 새 결과를 사용합니다.
- 이미 실행한 결과가 있으면 `larc_graph_data/dual_final_metrics.csv`와 `larc_graph_data/dual_overhead_metrics.csv`를 자동으로 읽습니다.


In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titleweight": "bold",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
    "legend.frameon": False,
})


In [ ]:
# 실행 및 파일 경로 설정
RUN_SOURCE_EXPERIMENT = False
SOURCE_NOTEBOOK = Path("Llama3_2_1B_Dual_ScoreOutput_LARC_K3V2_K2V2_NLL_PPL_ONLY.ipynb")
DATA_DIR = Path("larc_graph_data")
FIGURE_DIR = Path("larc_graph_outputs")
RESULTS_CSV = DATA_DIR / "dual_final_metrics.csv"
OVERHEAD_CSV = DATA_DIR / "dual_overhead_metrics.csv"

# 보고서에 이미 기재된 Table 1 수치입니다. 새 실험을 실행하거나 CSV를 넣으면 자동으로 대체됩니다.
REPORT_RESULTS = pd.DataFrame([
    {"label": "fp_baseline", "K_bits": 16, "V_bits": 16, "payload_eff_bits": 16.0, "nll": 0.369444, "ppl": 1.446930, "tokens_per_s": 21150.54},
    {"label": "dual_K3V2_base", "K_bits": 3, "V_bits": 2, "payload_eff_bits": 2.5, "nll": 1.128200, "ppl": 3.090089, "tokens_per_s": 6282.82},
    {"label": "dual_K3V2_ScoreOutput_LARC", "K_bits": 3, "V_bits": 2, "payload_eff_bits": 2.5, "nll": 0.502232, "ppl": 1.652405, "tokens_per_s": 4978.23},
    {"label": "dual_K2V2_base", "K_bits": 2, "V_bits": 2, "payload_eff_bits": 2.0, "nll": 4.168502, "ppl": 64.618605, "tokens_per_s": 6433.24},
    {"label": "dual_K2V2_ScoreOutput_LARC", "K_bits": 2, "V_bits": 2, "payload_eff_bits": 2.0, "nll": 1.185225, "ppl": 3.271423, "tokens_per_s": 5053.36},
])

# 초안에 명시된 Dual-LARC 파라미터 메모리입니다. 단위는 MiB(FP16)입니다.
REPORT_LARC_FIXED_MB = {"K3V2": 4.0025, "K2V2": 5.0206}
MODEL_SHAPE = {"num_layers": 16, "num_kv_heads": 8, "head_dim": 64}


In [ ]:
# 원본 실험 결과 불러오기
def run_source_notebook(path):
    if not path.exists():
        raise FileNotFoundError(f"원본 노트북을 찾을 수 없습니다: {path}")
    notebook = json.loads(path.read_text(encoding="utf-8"))
    for cell_index, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell["source"])
        exec(compile(source, f"{path.name}:cell-{cell_index}", "exec"), globals())


def make_overhead_snapshot():
    rows = []
    for mode, k_bits, v_bits in [("K3V2", 3, 2), ("K2V2", 2, 2)]:
        bits = (k_bits + v_bits) / 2
        for context_length in [512, 1024, 2048, 4096, 8192, 16384, 32768]:
            scalars = (MODEL_SHAPE["num_layers"] * 2 * MODEL_SHAPE["num_kv_heads"] * context_length * MODEL_SHAPE["head_dim"])
            fp16_mb = scalars * 16 / 8 / 1024**2
            payload_mb = scalars * bits / 8 / 1024**2
            larc_mb = REPORT_LARC_FIXED_MB[mode]
            rows.append({
                "mode": mode,
                "K_bits": k_bits,
                "V_bits": v_bits,
                "T_context": context_length,
                "payload_effective_bits": bits,
                "payload_compression_vs_fp16": 16 / bits,
                "original_fp16_KV_MB": fp16_mb,
                "KV_payload_MB": payload_mb,
                "Dual_LARC_fixed_MB_fp16": larc_mb,
                "total_fixed_overhead_MB": larc_mb,
                "total_memory_with_larc_MB": payload_mb + larc_mb,
                "fixed_overhead_pct_of_KV_payload": 100 * larc_mb / payload_mb,
            })
    return pd.DataFrame(rows)


if RUN_SOURCE_EXPERIMENT:
    run_source_notebook(SOURCE_NOTEBOOK)

if "dual_final_df" in globals() and "dual_overhead_df" in globals():
    raw_results = dual_final_df.copy()
    raw_overhead = dual_overhead_df.copy()
    data_source = "원본 실험 실행 결과"
elif RESULTS_CSV.exists() and OVERHEAD_CSV.exists():
    raw_results = pd.read_csv(RESULTS_CSV)
    raw_overhead = pd.read_csv(OVERHEAD_CSV)
    data_source = "저장된 실험 CSV"
else:
    raw_results = REPORT_RESULTS.copy()
    raw_overhead = make_overhead_snapshot()
    data_source = "초안 표 수치"

print(f"사용 수치: {data_source}")


In [ ]:
# 결과 열 이름과 실험 라벨을 정리합니다.
REQUIRED_RESULT_COLUMNS = {"label", "K_bits", "V_bits", "payload_eff_bits", "nll", "ppl", "tokens_per_s"}
missing_columns = REQUIRED_RESULT_COLUMNS - set(raw_results.columns)
if missing_columns:
    raise ValueError(f"최종 결과에 필요한 열이 없습니다: {sorted(missing_columns)}")

LABEL_INFO = {
    "fp_baseline": ("FP", "FP baseline", "FP"),
    "dual_K3V2_base": ("K3V2", "K3V2 base", "base"),
    "dual_K3V2_ScoreOutput_LARC": ("K3V2", "K3V2 + LARC", "LARC"),
    "dual_K2V2_base": ("K2V2", "K2V2 base", "base"),
    "dual_K2V2_ScoreOutput_LARC": ("K2V2", "K2V2 + LARC", "LARC"),
}
DISPLAY_ORDER = [
    "fp_baseline",
    "dual_K3V2_base",
    "dual_K3V2_ScoreOutput_LARC",
    "dual_K2V2_base",
    "dual_K2V2_ScoreOutput_LARC",
]

metrics_df = raw_results.copy()
metrics_df = metrics_df[metrics_df["label"].isin(DISPLAY_ORDER)].copy()
metrics_df["display_order"] = metrics_df["label"].map({label: index for index, label in enumerate(DISPLAY_ORDER)})
metrics_df = metrics_df.sort_values("display_order").reset_index(drop=True)
metrics_df[["mode", "display_label", "variant"]] = metrics_df["label"].apply(lambda label: pd.Series(LABEL_INFO[label]))
metrics_df["payload_compression_vs_fp16"] = 16 / metrics_df["payload_eff_bits"]

fp_row = metrics_df.loc[metrics_df["label"] == "fp_baseline"].iloc[0]
metrics_df["nll_delta_vs_fp"] = metrics_df["nll"] - fp_row["nll"]
metrics_df["ppl_delta_vs_fp"] = metrics_df["ppl"] - fp_row["ppl"]
metrics_df["ppl_ratio_vs_fp"] = metrics_df["ppl"] / fp_row["ppl"]
metrics_df["ppl_gap_pct_vs_fp"] = 100 * (metrics_df["ppl_ratio_vs_fp"] - 1)

display(metrics_df[["display_label", "payload_eff_bits", "payload_compression_vs_fp16", "nll", "ppl", "tokens_per_s"]])


In [ ]:
# 양자화 기준선 대비 복구량을 계산합니다.
recovery_rows = []
for mode in ["K3V2", "K2V2"]:
    base = metrics_df[(metrics_df["mode"] == mode) & (metrics_df["variant"] == "base")].iloc[0]
    larc = metrics_df[(metrics_df["mode"] == mode) & (metrics_df["variant"] == "LARC")].iloc[0]
    recovery_rows.append({
        "mode": mode,
        "payload_bits": larc["payload_eff_bits"],
        "compression_vs_fp16": larc["payload_compression_vs_fp16"],
        "base_nll": base["nll"],
        "larc_nll": larc["nll"],
        "nll_recovery": base["nll"] - larc["nll"],
        "base_ppl": base["ppl"],
        "larc_ppl": larc["ppl"],
        "ppl_recovery": base["ppl"] - larc["ppl"],
        "ppl_gap_recovered_pct": 100 * (base["ppl"] - larc["ppl"]) / (base["ppl"] - fp_row["ppl"]),
        "ppl_gap_to_fp_after_larc_pct": 100 * (larc["ppl"] / fp_row["ppl"] - 1),
    })

recovery_df = pd.DataFrame(recovery_rows)
display(recovery_df)

overhead_df = raw_overhead.copy()
if "mode" in overhead_df.columns:
    overhead_df["mode"] = overhead_df["mode"].str.replace("_Dual", "", regex=False)
if "total_memory_with_larc_MB" not in overhead_df.columns:
    overhead_df["total_memory_with_larc_MB"] = overhead_df["KV_payload_MB"] + overhead_df["total_fixed_overhead_MB"]
overhead_df = overhead_df.sort_values(["mode", "T_context"]).reset_index(drop=True)


In [ ]:
# 표와 그래프 파일을 저장할 폴더를 만듭니다.
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

metrics_df.to_csv(DATA_DIR / "graph_metrics.csv", index=False)
recovery_df.to_csv(DATA_DIR / "recovery_metrics.csv", index=False)
overhead_df.to_csv(DATA_DIR / "context_memory_metrics.csv", index=False)
raw_results.to_csv(RESULTS_CSV, index=False)
raw_overhead.to_csv(OVERHEAD_CSV, index=False)

figure_values = {
    "data_source": data_source,
    "model": "meta-llama/Llama-3.2-1B",
    "metrics": metrics_df.to_dict(orient="records"),
    "recovery": recovery_df.to_dict(orient="records"),
}
with (DATA_DIR / "figure_values.json").open("w", encoding="utf-8") as file:
    json.dump(figure_values, file, ensure_ascii=False, indent=2, default=lambda value: value.item() if hasattr(value, "item") else str(value))


In [ ]:
# 그래프 1: NLL과 PPL 비교
labels = metrics_df["display_label"].tolist()
x = np.arange(len(metrics_df))
colors = ["#4C566A", "#D08770", "#5E81AC", "#BF616A", "#88C0D0"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
axes[0].bar(x, metrics_df["nll"], color=colors)
axes[0].set_title("NLL 비교")
axes[0].set_ylabel("NLL")
axes[0].set_xticks(x, labels, rotation=20, ha="right")
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(x, metrics_df["ppl"], color=colors)
axes[1].set_title("PPL 비교 (로그 축)")
axes[1].set_ylabel("PPL")
axes[1].set_yscale("log")
axes[1].set_xticks(x, labels, rotation=20, ha="right")
axes[1].grid(axis="y", alpha=0.25, which="both")

fig.savefig(FIGURE_DIR / "01_quality_nll_ppl.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 그래프 2: LARC 적용 전후 PPL 복구
fig, ax = plt.subplots(figsize=(7.6, 4.8), constrained_layout=True)
x = np.arange(len(recovery_df))
width = 0.34
ax.bar(x - width / 2, recovery_df["base_ppl"], width, label="quantized base", color="#D08770")
ax.bar(x + width / 2, recovery_df["larc_ppl"], width, label="ScoreOutput LARC", color="#5E81AC")
ax.axhline(fp_row["ppl"], color="#4C566A", linestyle="--", linewidth=1.5, label="FP baseline")
ax.set_yscale("log")
ax.set_ylabel("PPL (로그 축)")
ax.set_xticks(x, recovery_df["mode"])
ax.set_title("양자화 기준선 대비 PPL 복구")
ax.grid(axis="y", alpha=0.25, which="both")
ax.legend()
for index, row in recovery_df.iterrows():
    y = max(row["base_ppl"], row["larc_ppl"])
    ax.text(index, y * 1.23, f"-{row['ppl_recovery']:.3f}", ha="center", va="bottom", fontsize=10)

fig.savefig(FIGURE_DIR / "02_ppl_recovery.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 그래프 3: 문맥 길이에 따른 KV 메모리와 고정 오버헤드
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
mode_colors = {"K3V2": "#5E81AC", "K2V2": "#BF616A"}
for mode, group in overhead_df.groupby("mode", sort=False):
    color = mode_colors.get(mode, "#4C566A")
    axes[0].plot(group["T_context"], group["original_fp16_KV_MB"], color=color, linestyle=":", linewidth=2, label=f"{mode} FP16 KV")
    axes[0].plot(group["T_context"], group["total_memory_with_larc_MB"], color=color, linewidth=2.5, label=f"{mode} payload + LARC")
    axes[1].plot(group["T_context"], group["fixed_overhead_pct_of_KV_payload"], color=color, linewidth=2.5, label=mode)

for ax in axes:
    ax.set_xscale("log", base=2)
    ax.grid(alpha=0.25)
    ax.legend()
axes[0].set_title("문맥 길이별 KV 메모리")
axes[0].set_xlabel("context length")
axes[0].set_ylabel("MiB")
axes[1].set_title("고정 오버헤드 비율")
axes[1].set_xlabel("context length")
axes[1].set_ylabel("payload 대비 고정 오버헤드 (%)")

fig.savefig(FIGURE_DIR / "03_context_memory.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 그래프 4: 압축률과 품질의 관계
fig, ax = plt.subplots(figsize=(7.6, 5.0), constrained_layout=True)
marker_map = {"FP": "s", "base": "o", "LARC": "D"}
color_map = {"FP": "#4C566A", "K3V2": "#5E81AC", "K2V2": "#BF616A"}
for _, row in metrics_df.iterrows():
    ax.scatter(row["payload_compression_vs_fp16"], row["ppl"], s=105, marker=marker_map[row["variant"]], color=color_map[row["mode"]], zorder=3)
    ax.annotate(row["display_label"], (row["payload_compression_vs_fp16"], row["ppl"]), xytext=(7, 7), textcoords="offset points", fontsize=9)
ax.set_yscale("log")
ax.set_xlabel("payload compression vs FP16 (x)")
ax.set_ylabel("PPL (로그 축)")
ax.set_title("압축률과 언어모델 품질")
ax.grid(alpha=0.25, which="both")
fig.savefig(FIGURE_DIR / "04_compression_quality.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 그래프 5: 측정 처리량과 품질의 관계
fig, ax = plt.subplots(figsize=(7.6, 5.0), constrained_layout=True)
for _, row in metrics_df.iterrows():
    ax.scatter(row["tokens_per_s"], row["ppl"], s=105, marker=marker_map[row["variant"]], color=color_map[row["mode"]], zorder=3)
    ax.annotate(row["display_label"], (row["tokens_per_s"], row["ppl"]), xytext=(7, 7), textcoords="offset points", fontsize=9)
ax.set_yscale("log")
ax.set_xlabel("measured tokens/s")
ax.set_ylabel("PPL (로그 축)")
ax.set_title("측정 처리량과 품질")
ax.grid(alpha=0.25, which="both")
fig.savefig(FIGURE_DIR / "05_speed_quality.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 저장된 파일을 확인합니다.
print(f"수치 원본: {data_source}")
print("\nCSV / JSON")
for path in sorted(DATA_DIR.glob("*")):
    print("-", path)
print("\n그래프")
for path in sorted(FIGURE_DIR.glob("*.png")):
    print("-", path)
print("\n핵심 복구 수치")
display(recovery_df[["mode", "nll_recovery", "ppl_recovery", "ppl_gap_recovered_pct", "ppl_gap_to_fp_after_larc_pct"]])
